In [19]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from scipy.stats import pearsonr

from skimage.metrics import (
    structural_similarity as ssim,
    peak_signal_noise_ratio as psnr,
)

# ============================================================
# Leitura dos arquivos
# ============================================================

def load_predictions(csv_path: str) -> pd.DataFrame:

    df = pd.read_csv(csv_path)

    df["year"] = df["year"].astype(int)
    df["month"] = df["month"].astype(int)

    df["volume"] = (
        df["volume_m2"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    df["volume"] = pd.to_numeric(
        df["volume"],
        errors="coerce"
    )

    return df


def load_ground_truth(csv_path: str) -> pd.DataFrame:
    """
    Carrega o CSV contendo os volumes observados.
    """

    df = pd.read_csv(csv_path)

    df["Data da Medição"] = pd.to_datetime(
        df["Data da Medição"],
        dayfirst=True,
    )
    
    df["Volume Útil (hm³)"] = pd.to_numeric(
        df["Volume Útil (hm³)"],
        errors="coerce"
    )

    return df


# ============================================================
# Filtro
# ============================================================

def filter_predictions(
    df: pd.DataFrame,
    location: str,
    segmentation_method: str,
    threshold: float,
) -> pd.DataFrame:
    """
    Filtra um conjunto específico de experimentos.
    """

    return df[
        (df["location"] == location)
        & (df["segmentation_method"] == segmentation_method)
        & (df["threshold"] == threshold)
    ].copy()


# ============================================================
# Média mensal
# ============================================================

def monthly_predictions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula a média mensal dos volumes previstos.
    """

    return (
        df.groupby(
            ["year", "month"],
            as_index=False
        )
        .agg(
            volume=("volume_m2", "mean")
        )
    )


def monthly_ground_truth(df):

    df = df.copy()

    df["Volume Útil (hm³)"] = pd.to_numeric(
        df["Volume Útil (hm³)"],
        errors="coerce"
    )

    df = df.dropna(subset=["Volume Útil (hm³)"])

    df["year"] = df["Data da Medição"].dt.year
    df["month"] = df["Data da Medição"].dt.month

    return (
        df.groupby(["year", "month"], as_index=False)
          .agg(
              ground_truth_volume=("Volume Útil (hm³)", "mean")
          )
    )


# ============================================================
# Associação
# ============================================================

def match_monthly(
    predictions: pd.DataFrame,
    ground_truth: pd.DataFrame,
) -> pd.DataFrame:
    """
    Junta as séries mensais.
    """

    return predictions.merge(
        ground_truth,
        on=["year", "month"],
        how="inner",
    )


# ============================================================
# Métricas de regressão
# ============================================================

def regression_metrics(df: pd.DataFrame) -> dict:

    y_true = df["ground_truth_volume"].to_numpy(dtype=float)
    y_pred = df["volume"].to_numpy(dtype=float)

    mae = mean_absolute_error(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    r2 = r2_score(y_true, y_pred)

    pearson, _ = pearsonr(y_true, y_pred)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Pearson": pearson,
    }


# ============================================================
# Métricas de similaridade
# ============================================================

def similarity_metrics(df: pd.DataFrame) -> dict:

    y_true = df["ground_truth_volume"].to_numpy(dtype=np.float64)
    y_pred = df["volume"].to_numpy(dtype=np.float64)

    data_range = max(
        y_true.max(),
        y_pred.max(),
    ) - min(
        y_true.min(),
        y_pred.min(),
    )

    if data_range == 0:
        data_range = 1.0

    ssim_value = ssim(
        y_true,
        y_pred,
        data_range=data_range,
    )

    psnr_value = psnr(
        y_true,
        y_pred,
        data_range=data_range,
    )

    return {
        "SSIM": ssim_value,
        "PSNR": psnr_value,
    }


# ============================================================
# Todas as métricas
# ============================================================

def compute_metrics(df: pd.DataFrame) -> dict:

    metrics = {}

    metrics.update(regression_metrics(df))
    metrics.update(similarity_metrics(df))

    return metrics


# ============================================================
# Pipeline completo
# ============================================================

def evaluate(
    prediction_csv: str,
    ground_truth_csv: str,
    location: str,
    segmentation_method: str,
    threshold: float,
):

    predictions = load_predictions(prediction_csv)

    ground_truth = load_ground_truth(ground_truth_csv)

    # predictions = filter_predictions(
    #     predictions,
    #     location,
    #     segmentation_method,
    #     threshold,
    # )

    predictions = monthly_predictions(predictions)

    ground_truth = monthly_ground_truth(ground_truth)

    comparison = match_monthly(
        predictions,
        ground_truth,
    )

    metrics = compute_metrics(comparison)

    return comparison, metrics

In [20]:
comparison, metrics = evaluate(
    prediction_csv="D:/GeoPipe/notebooks/data/volume/gramame_mamuaba_reservatorio_vggunet_fmask/df_volumes_trh_0.05.csv",
    ground_truth_csv="D:/GeoPipe/notebooks/data/ana/gramame_mamuaba_ana.csv",
    location="gramame_mamuaba",
    segmentation_method="arianet",
    threshold=0.5,
)

comparison.head()

metrics

{'MAE': 3.2109877597162844,
 'MSE': 12.132677244646056,
 'RMSE': 3.4831992829360274,
 'MAPE': 7.555324171638256,
 'R2': 0.5045823458436195,
 'Pearson': 0.7455435770779534,
 'SSIM': 0.6238277484460596,
 'PSNR': 10.340279626448094}